# Transport, spallation, and adiabatic losses
Analyze the two outputs of `transport_spallation_loss.cpp` using NumPy and Matplotlib.

- `transport_spallation_loss.bin`: native-endian float64 records `(x, y, z, E, species_id)`; positions in pc, **total nuclear energy** in GeV. Destroyed particles have ID −1 and retain their last position/energy in initial particle order.
- `earth_arrivals.bin`: native-endian float64 records `(E, species_id, time_yr)` (24 bytes), with total nuclear energy in GeV and elapsed propagation time in years. Every step ending inside the solar-ring region contributes a record; the same particle can contribute many times. Arrival times refer to step endpoints; no particle ID is saved. Regenerate both binaries with the updated C++ program before running this notebook; legacy two-column arrival files are incompatible.

The files contain no metadata. Parameters below describe the current source and must match the run. The simulation uses constant hydrogen density, a linear vertical wind, adiabatic cooling, and single-residual spallation without an escape boundary. It does not include continuous hadronic energy losses. Run from `output/` or the project directory.
The new energy convention is converted to total energy per nucleon using the recorded isotope mass number. Legacy GeV/n binaries require a different reader. Injection is uniform in area inside R = 10 kpc and uniform in z between −100 and +100 pc (200 pc full thickness, as implemented).


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.dpi': 120})
data_dir = Path.cwd()
if not (data_dir / 'transport_spallation_loss.bin').exists():
    data_dir = data_dir / 'output'

# Run parameters: edit for binaries produced with other settings.
E_min, E_max = 10.0, 1e6  # injected total energy per nucleon, GeV/n
T_end = 1e8               # yr
a = 1e-8                  # div(u), yr^-1; u_z = a*z
m_p = 0.938272            # nucleon rest energy, GeV
nH = 0.1                 # cm^-3
expected_particles = 210000
species_names = ['O16', 'N15', 'N14', 'C13', 'C12',
                 'B11', 'B10', 'Be10', 'Be9', 'Be7']
mass_numbers = np.array([16, 15, 14, 13, 12, 11, 10, 10, 9, 7])
charges = np.array([8, 7, 7, 6, 6, 5, 5, 4, 4, 4])
element_ids = {'O': [0], 'N': [1, 2], 'C': [3, 4],
               'B': [5, 6], 'Be': [7, 8, 9]}


def load_records(filename, columns):
    path = data_dir / filename
    if path.stat().st_size % (8 * columns):
        raise ValueError(f'{path}: incomplete float64 record')
    return np.fromfile(path, dtype='=f8').reshape(-1, columns)


particles = load_records('transport_spallation_loss.bin', 5)
arrivals = load_records('earth_arrivals.bin', 3)
x, y, z, E, species = particles.T
E_arr, species_arr, time_arr = arrivals.T
if np.any((time_arr < 0) | (time_arr > T_end)) or not np.all(np.isin(species_arr, np.arange(10))):
    raise ValueError('Invalid arrival records; regenerate earth_arrivals.bin in the new three-column format.')
tracked = species >= 0
N = len(E)
# Removed records retain the last parent's energy but not its mass number.
E_n = np.full(N, np.nan)
E_n[tracked] = E[tracked] / mass_numbers[species[tracked].astype(int)]
E_arr_n = E_arr / mass_numbers[species_arr.astype(int)]
print(f'Data directory: {data_dir.resolve()}')
print(f'Final records: {N:,} (configured injection: {expected_particles:,})')
print(f'Tracked: {tracked.sum():,}; destroyed: {(~tracked).sum():,}')
print(f'Earth-arrival step samples: {len(arrivals):,}')
if len(time_arr):
    print('Arrival time min / median / max [Myr]:', np.percentile(time_arr / 1e6, [0, 50, 100]))
print('Recorded total nuclear energy range [GeV]:', E.min(), E.max())
print('Tracked total energy per nucleon range [GeV/n]:', E_n[tracked].min(), E_n[tracked].max())

## Final abundances and sampled arrival composition
Final abundances count particles at the end of transport (or removal). Arrival composition counts step samples accumulated throughout propagation. These populations have different meanings; their normalizations should not be compared as survival probabilities.

In [ ]:
ids = np.arange(-1, 10)
final_counts = np.array([np.count_nonzero(species == i) for i in ids])
arrival_counts = np.array([np.count_nonzero(species_arr == i) for i in range(10)])
print(f'{"Species":>10} {"Final particles":>16} {"Arrival samples":>17}')
for label, count, arrival_count in zip(['Destroyed'] + species_names,
                                      final_counts, np.r_[0, arrival_counts]):
    print(f'{label:>10} {count:16,d} {arrival_count:17,d}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(['Removed'] + species_names, final_counts / N)
axes[0].set_ylabel('Fraction of injected particles')
axes[0].set_title('Final state')
axes[1].bar(species_names, arrival_counts / max(len(arrivals), 1))
axes[1].set_ylabel('Fraction of arrival step samples')
axes[1].set_title('Solar-ring samples')
for ax in axes:
    ax.tick_params(axis='x', rotation=60)
fig.tight_layout()
plt.show()

## Energy spectra
The injected grid is uniform in log total energy per nucleon. Spectra show counts per logarithmic interval, without source reweighting or flux normalization. Tracked and arrival energies are divided by the recorded isotope mass number. Fragmentation preserves energy per nucleon in this model.

Removed records have no recoverable mass number and are excluded from per-nucleon spectra. Their stored energies belong to the last tracked parent at removal, rather than the final simulation time.


In [ ]:
E0 = np.geomspace(E_min, E_max, N)  # output preserves injection order
energy_low = min(E_n[tracked].min(), E_arr_n.min() if len(E_arr_n) else E_min, E_min)
energy_high = max(E_n[tracked].max(), E_arr_n.max() if len(E_arr_n) else E_max, E_max)
bins = np.geomspace(energy_low * (1 - 1e-12), energy_high * (1 + 1e-12), 31)
Ec = np.sqrt(bins[:-1] * bins[1:])
dlnE = np.diff(np.log(bins))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for values, label in [(E0, 'Injection'), (E_n[tracked], 'Final tracked')]:
    counts, _ = np.histogram(values, bins)
    axes[0].stairs(counts / dlnE, bins, label=label)
for element, isotope_ids in element_ids.items():
    counts, _ = np.histogram(E_arr_n[np.isin(species_arr, isotope_ids)], bins)
    axes[1].stairs(counts / dlnE, bins, label=element)
for ax in axes:
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Total energy per nucleon [GeV/n]')
    ax.legend()
axes[0].set_ylabel(r'$dN/d\ln E$ [particles]')
axes[1].set_ylabel(r'$dN_{\mathrm{sample}}/d\ln E$ [step samples]')
axes[0].set_title('Injection and final records')
axes[1].set_title('Solar-ring samples by element')
fig.tight_layout()
plt.show()

## Boron-to-carbon ratio
Sum B10+B11 and C12+C13 in the same total-energy-per-nucleon bins. Empty carbon bins are omitted. The final-state ratio and the accumulated arrival-sample ratio are distinct diagnostics. Arrival samples are correlated within each particle, so independent Poisson error bars would not describe their uncertainty.

In [ ]:
def element_counts(energies, species_ids, element):
    mask = np.isin(species_ids, element_ids[element])
    return np.histogram(energies[mask], bins)[0]


fig, ax = plt.subplots(figsize=(7, 4))
for energies, species_ids, label in [(E_n, species, 'Final tracked particles'),
                                    (E_arr_n, species_arr, 'Solar-ring step samples')]:
    B = element_counts(energies, species_ids, 'B')
    C = element_counts(energies, species_ids, 'C')
    ratio = np.divide(B, C, out=np.full(len(C), np.nan), where=C > 0)
    ax.semilogx(Ec, ratio, 'o-', label=label)
    print(f'{label}: total B = {B.sum():,}, total C = {C.sum():,}')
ax.set_xlabel('Total energy per nucleon [GeV/n]')
ax.set_ylabel('B/C [count ratio]')
ax.set_yscale("log")
ax.legend()
fig.tight_layout()
plt.show()

## Adiabatic cooling
Use total energy per nucleon $\epsilon=E/A$. Since fragmentation preserves $\epsilon$ and all nuclei have rest energy $A m_p$, constant divergence gives
$$\epsilon(t)=\sqrt{m_p^2+(\epsilon_0^2-m_p^2)e^{-2at/3}}.$$
Tracked particles evolve until $T_{\rm end}$, so their recorded energy per nucleon can be compared with the injected grid in record order, even after fragmentation. Removed records lack the last parent's mass number, so their cooling times cannot be inferred reliably. Arrival records cannot be linked to final particles.


In [ ]:
E_reference = np.sqrt(m_p**2 + (E0**2 - m_p**2) * np.exp(-2*a*T_end/3))
p0 = np.sqrt(E0[tracked]**2 - m_p**2)
p_final = np.sqrt(E_n[tracked]**2 - m_p**2)
t_inferred = -3.0 / a * np.log(p_final / p0)  # yr; a != 0
residual = (E_n[tracked] - E_reference[tracked]) / E_reference[tracked]
print('Max |relative energy residual|, tracked:', np.max(np.abs(residual)))
print('Inferred tracked time min / median / max [Myr]:',
      np.percentile(t_inferred / 1e6, [0, 50, 100]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
stride = max(1, tracked.sum() // 15000)
idx = np.flatnonzero(tracked)[::stride]
axes[0].loglog(E0[idx], E_n[idx], '.', ms=1, label='Final tracked')
axes[0].loglog(E0, E_reference, 'k--', label='Cooling at T_end')
axes[0].set_xlabel('Injected total energy per nucleon [GeV/n]')
axes[0].set_ylabel('Final total energy per nucleon [GeV/n]')
axes[0].legend(markerscale=4)
axes[1].semilogx(E0[tracked], residual, '.', ms=1)
axes[1].axhline(0, color='k', ls='--')
axes[1].set_xlabel('Injected total energy per nucleon [GeV/n]')
axes[1].set_ylabel('(Recorded - reference) / reference')
fig.tight_layout()
plt.show()


## Spatial distribution of tracked particles
Positions below use kpc. Removed particles are excluded because their coordinates freeze at destruction. The solar-ring selection is centered on $R=8$ kpc, $z=0$, with an energy-dependent radius $\lambda(\mathcal R)=2(\mathcal R/10\,\mathrm{GV})^{1/3}$ pc, where $\mathcal R[\mathrm{GV}]=\sqrt{E^2-(A m_p)^2}/Z$ for total nuclear energy $E$ in GeV; it is a ring rather than a point detector. Final positions do not reconstruct arrival trajectories.

In [ ]:
R = np.hypot(x, y) / 1000.0
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
if tracked.any():
    image = axes[0].hexbin(x[tracked] / 1000, y[tracked] / 1000,
                          gridsize=70, bins='log', mincnt=1)
    fig.colorbar(image, ax=axes[0], label='Tracked particles per hexagon')
    image = axes[1].hexbin(R[tracked], z[tracked] / 1000,
                          gridsize=70, bins='log', mincnt=1)
    fig.colorbar(image, ax=axes[1], label='Tracked particles per hexagon')
axes[0].set(xlabel='x [kpc]', ylabel='y [kpc]', title='Final Galactic plane')
axes[0].set_aspect('equal')
axes[1].plot(8.0, 0.0, 'r+', ms=10, label='Solar-ring center')
axes[1].set(xlabel='R [kpc]', ylabel='z [kpc]', title='Final cylindrical distribution')
axes[1].legend()
fig.tight_layout()
plt.show()